# KNN with scikit-learn - Lab

## Introduction

In this lab, you'll learn how to use scikit-learn's implementation of a KNN classifier on the classic Titanic dataset from Kaggle!
 

## Objectives

In this lab you will:

- Conduct a parameter search to find the optimal value for K 
- Use a KNN classifier to generate predictions on a real-world dataset 
- Evaluate the performance of a KNN model  


## Getting Started

Start by importing the dataset, stored in the `titanic.csv` file, and previewing it.

In [22]:
# Your code here
# Import pandas and set the standard alias 
import pandas as pd

# Import the data from 'titanic.csv' and store it in a pandas DataFrame 
raw_df = pd.read_csv('titanic.csv')

# Print the head of the DataFrame to ensure everything loaded correctly 
print(raw_df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [23]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


Great!  Next, you'll perform some preprocessing steps such as removing unnecessary columns and normalizing features.

## Preprocessing the data

Preprocessing is an essential component in any data science pipeline. It's not always the most glamorous task as might be an engaging data visual or impressive neural network, but cleaning and normalizing raw datasets is very essential to produce useful and insightful datasets that form the backbone of all data powered projects. This can include changing column types, as in: 


```python
df['col_name'] = df['col_name'].astype('int')
```
Or extracting subsets of information, such as: 

```python
import re
df['street'] = df['address'].map(lambda x: re.findall('(.*)?\n', x)[0])
```

> **Note:** While outside the scope of this particular lesson, **regular expressions** (mentioned above) are powerful tools for pattern matching! See the [regular expressions official documentation here](https://docs.python.org/3.6/library/re.html). 

Since you've done this before, you should be able to do this quite well yourself without much hand holding by now. In the cells below, complete the following steps:

1. Remove unnecessary columns (`'PassengerId'`, `'Name'`, `'Ticket'`, and `'Cabin'`) 
2. Convert `'Sex'` to a binary encoding, where female is `0` and male is `1` 
3. Detect and deal with any missing values in the dataset:  
    * For `'Age'`, replace missing values with the median age for the dataset  
    * For `'Embarked'`, drop the rows that contain missing values
4. One-hot encode categorical columns such as `'Embarked'` 
5. Store the target column, `'Survived'`, in a separate variable and remove it from the DataFrame  

While we always want to worry about data leakage, which is why we typically perform the split before the preprocessing, for this data set, we'll do some of the preprocessing first. The reason for this is that some of the values of the variables only have a handful of instances, and we want to make sure we don't lose any of them.

In [24]:
# Drop the unnecessary columns
df = raw_df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [25]:
# Convert Sex to binary encoding
df['Sex'] = df['Sex'].map({'female': 0, 'male': 1})
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,S
1,1,1,0,38.0,1,0,71.2833,C
2,1,3,0,26.0,0,0,7.9250,S
3,1,1,0,35.0,1,0,53.1000,S
4,0,3,1,35.0,0,0,8.0500,S


In [26]:
# Find the number of missing values in each column
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [27]:
# Impute the missing values in 'Age'
df['Age'] = df['Age'].fillna(df['Age'].median())
df.isna().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    2
dtype: int64

In [28]:
# Drop the rows missing values in the 'Embarked' column
df = df.dropna(subset=['Embarked'])
df.isna().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Sex       889 non-null    int64  
 3   Age       889 non-null    float64
 4   SibSp     889 non-null    int64  
 5   Parch     889 non-null    int64  
 6   Fare      889 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(5), object(1)
memory usage: 62.5+ KB


In [30]:
# To understand unique values in the 'Embarked' column
print(df['Embarked'].unique())

['S' 'C' 'Q']


In [31]:
# To understand and visualize unique values and their frequencies
print(df['Embarked'].value_counts())

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64


In [32]:
# One-hot encode the categorical columns
one_hot_df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
one_hot_df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,1,22.0,1,0,7.2500,False,True
1,1,1,0,38.0,1,0,71.2833,False,False
2,1,3,0,26.0,0,0,7.9250,False,True
3,1,1,0,35.0,1,0,53.1000,False,True
4,0,3,1,35.0,0,0,8.0500,False,True


In [33]:
# Assign the 'Survived' column to labels
labels = one_hot_df['Survived']

# Drop the 'Survived' column from one_hot_df
one_hot_df = one_hot_df.drop(columns=['Survived'])

## Create training and test sets

Now that you've preprocessed the data, it's time to split it into training and test sets. 

In the cell below:

* Import `train_test_split` from the `sklearn.model_selection` module 
* Use `train_test_split()` to split the data into training and test sets, with a `test_size` of `0.25`. Set the `random_state` to 42 

In [34]:
# Import train_test_split 
from sklearn.model_selection import train_test_split

# Split the data
X_train, X_test, y_train, y_test = train_test_split(one_hot_df, labels, test_size=0.25, random_state=42)

## Normalizing the data

The final step in your preprocessing efforts for this lab is to **_normalize_** the data. We normalize **after** splitting our data into training and test sets. This is to avoid information "leaking" from our test set into our training set (read more about data leakage [here](https://machinelearningmastery.com/data-leakage-machine-learning/) ). Remember that normalization (also sometimes called **_Standardization_** or **_Scaling_**) means making sure that all of your data is represented at the same scale. The most common way to do this is to convert all numerical values to z-scores. 

Since KNN is a distance-based classifier, if data is in different scales, then larger scaled features have a larger impact on the distance between points.

To scale your data, use `StandardScaler` found in the `sklearn.preprocessing` module. 

In the cell below:

* Import and instantiate `StandardScaler` 
* Use the scaler's `.fit_transform()` method to create a scaled version of the training dataset  
* Use the scaler's `.transform()` method to create a scaled version of the test dataset  
* The result returned by `.fit_transform()` and `.transform()` methods will be numpy arrays, not a pandas DataFrame. Create a new pandas DataFrame out of this object called `scaled_df`. To set the column names back to their original state, set the `columns` parameter to `one_hot_df.columns` 
* Print the head of `scaled_df` to ensure everything worked correctly 

In [35]:
# Import StandardScaler
from sklearn.preprocessing import StandardScaler

# Instantiate StandardScaler
scaler = StandardScaler()

# Transform the training and test sets
scaled_data_train = scaler.fit_transform(X_train)
scaled_data_test = scaler.transform(X_test)

# Convert into a DataFrame
scaled_df_train = pd.DataFrame(scaled_data_train, columns=one_hot_df.columns)
scaled_df_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0.815528,-1.390655,-0.575676,-0.474917,-0.480663,-0.500108,-0.311768,0.620174
1,-0.386113,-1.390655,1.550175,-0.474917,-0.480663,-0.435393,-0.311768,0.620174
2,-0.386113,0.719086,-0.120137,-0.474917,-0.480663,-0.644473,-0.311768,0.620174
3,-1.587755,0.719086,-0.120137,-0.474917,-0.480663,-0.115799,-0.311768,0.620174
4,0.815528,-1.390655,-1.107139,0.413551,-0.480663,-0.356656,-0.311768,-1.612452


You may have noticed that the scaler also scaled our binary/one-hot encoded columns, too! Although it doesn't look as pretty, this has no negative effect on the model. Each 1 and 0 have been replaced with corresponding decimal values, but each binary column still only contains 2 values, meaning the overall information content of each column has not changed.

## Fit a KNN model

Now that you've preprocessed the data it's time to train a KNN classifier and validate its accuracy. 

In the cells below:

* Import `KNeighborsClassifier` from the `sklearn.neighbors` module 
* Instantiate the classifier. For now, you can just use the default parameters  
* Fit the classifier to the training data/labels
* Use the classifier to generate predictions on the test data. Store these predictions inside the variable `test_preds` 

In [36]:
# Import KNeighborsClassifier
from sklearn.neighbors import KNeighborsClassifier

# Instantiate KNeighborsClassifier
clf = KNeighborsClassifier()

# Fit the classifier
clf.fit(scaled_data_train, y_train)

# Predict on the test set
test_preds = clf.predict(scaled_data_test)

## Evaluate the model

Now, in the cells below, import all the necessary evaluation metrics from `sklearn.metrics` and complete the `print_metrics()` function so that it prints out **_Precision, Recall, Accuracy, and F1-Score_** when given a set of `labels` (the true values) and `preds` (the models predictions). 

Finally, use `print_metrics()` to print the evaluation metrics for the test predictions stored in `test_preds`, and the corresponding labels in `y_test`. 

In [37]:
# Your code here 
# Import the necessary functions
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

def print_metrics(labels, preds):
    print(f"Accuracy:  {accuracy_score(labels, preds):.4f}")
    print(f"Precision: {precision_score(labels, preds):.4f}")
    print(f"Recall:    {recall_score(labels, preds):.4f}")
    print(f"F1-Score:  {f1_score(labels, preds):.4f}")

# Call the function using your test data and predictions
print_metrics(y_test, test_preds)


Accuracy:  0.7892
Precision: 0.7059
Recall:    0.7317
F1-Score:  0.7186


In [39]:
# Complete the function
def print_metrics(labels, preds):
    print("Precision Score: {}".format(precision_score(labels, preds)))
    print("Recall Score: {}".format(recall_score(labels, preds)))
    print("Accuracy Score: {}".format(accuracy_score(labels, preds)))
    print("F1 Score: {}".format(f1_score(labels, preds)))
    
print_metrics(y_test, test_preds)

Precision Score: 0.7058823529411765
Recall Score: 0.7317073170731707
Accuracy Score: 0.7892376681614349
F1 Score: 0.718562874251497


> Interpret each of the metrics above, and explain what they tell you about your model's capabilities. If you had to pick one score to best describe the performance of the model, which would you choose? Explain your answer.

Write your answer below this line: 


Interpretation of the Metrics
Accuracy Score (78.92%): This tells us that your model correctly predicted whether a passenger lived or died about 79% of the time across the entire test set. While a solid baseline, accuracy can sometimes be misleading if your dataset is highly imbalanced (e.g., if way more people died than survived).

Precision Score (70.59%): Out of all the passengers your model predicted would survive, 70.59% of them actually did. The remaining ~29.4% were "False Positives"—passengers the model thought would make it, but unfortunately did not.

Recall Score (73.17%): Out of all the passengers who actually survived the shipwreck, your model successfully caught 73.17% of them. The remaining ~26.8% were "False Negatives"—survivors the model completely missed.

F1 Score (71.86%): This is the harmonic mean of your Precision and Recall. Because both Precision and Recall are in the low-70s, the F1 Score balances them out to give a realistic view of how well the model handles the positive class (the survivors).

Which score best describes the model's performance?
If forced to pick just one metric to evaluate this model, the F1 Score is the best choice.

Why?
In the Titanic dataset, survival is the minority class (more people tragically perished than survived). When dealing with unevenly distributed classes, standard Accuracy can artificially inflate your confidence.

Between Precision and Recall, neither one inherently completely outvalues the other in a historical analysis context like this (unlike in medicine, where high recall is vital to catch every sick patient). Because Precision and Recall are both tracking completely different types of errors, the F1 Score acts as the ultimate tiebreaker. A strong F1 Score proves that your model isn't achieving high precision by simply being afraid to guess "Survived," nor is it achieving high recall by recklessly guessing "Survived" for everyone. It gives you a single, honest look at the model's true balance.

## Improve model performance

While your overall model results should be better than random chance, they're probably mediocre at best given that you haven't tuned the model yet. For the remainder of this notebook, you'll focus on improving your model's performance. Remember that modeling is an **_iterative process_**, and developing a baseline out of the box model such as the one above is always a good start. 

First, try to find the optimal number of neighbors to use for the classifier. To do this, complete the `find_best_k()` function below to iterate over multiple values of K and find the value of K that returns the best overall performance. 

The function takes in six arguments:
* `X_train`
* `y_train`
* `X_test`
* `y_test`
* `min_k` (default is 1)
* `max_k` (default is 25)
    
> **Pseudocode Hint**:
1. Create two variables, `best_k` and `best_score`
1. Iterate through every **_odd number_** between `min_k` and `max_k + 1`. 
    1. For each iteration:
        1. Create a new `KNN` classifier, and set the `n_neighbors` parameter to the current value for k, as determined by the loop 
        1. Fit this classifier to the training data 
        1. Generate predictions for `X_test` using the fitted classifier 
        1. Calculate the **_F1-score_** for these predictions 
        1. Compare this F1-score to `best_score`. If better, update `best_score` and `best_k` 
1. Once all iterations are complete, print the best value for k and the F1-score it achieved 

In [ ]:
def find_best_k(X_train, y_train, X_test, y_test, min_k=1, max_k=25):
    # 1. Create variables to keep track of the best results
    best_k = None
    best_score = 0.0
    
    # 2. Iterate through every odd number between min_k and max_k
    for k in range(min_k, max_k + 1):
        if k % 2 != 0:  # Check if the number is odd
            
            # A. Create a new KNN classifier with the current k
            knn = KNeighborsClassifier(n_neighbors=k)
            
            # B. Fit this classifier to the training data 
            knn.fit(X_train, y_train)
            
            # C. Generate predictions for X_test
            preds = knn.predict(X_test)
            
            # D. Calculate the F1-score for these predictions
            score = f1_score(y_test, preds)
            
            # E. If this score is better than our previous best, update our variables
            if score > best_score:
                best_score = score
                best_k = k
                
    # 3. Print the best value for k and the F1-score it achieved
    print(f"Best K value: {best_k}")
    print(f"Best F1-Score achieved: {best_score:.4f}")
    
    return best_k

In [42]:
def find_best_k(X_train, y_train, X_test, y_test, min_k=1, max_k=25):

    # 1. Create variables to keep track of the best results
    best_k = None
    best_score = 0.0
    
    # 2. Iterate through every odd number between min_k and max_k
    for k in range(min_k, max_k + 1):
        if k % 2 != 0:  # Check if the number is odd
            
            # A. Create a new KNN classifier with the current k
            knn = KNeighborsClassifier(n_neighbors=k)
            
            # B. Fit this classifier to the training data 
            knn.fit(X_train, y_train)
            
            # C. Generate predictions for X_test
            preds = knn.predict(X_test)
            
            # D. Calculate the F1-score for these predictions
            score = f1_score(y_test, preds)
            
            # E. If this score is better than our previous best, update our variables
            if score > best_score:
                best_score = score
                best_k = k
                
    # 3. Print the best value for k and the F1-score it achieved
    print(f"Best K value: {best_k}")
    print(f"Best F1-Score achieved: {best_score:.4f}")
    
    return best_k


In [43]:
find_best_k(scaled_data_train, y_train, scaled_data_test, y_test)
# Expected Output:

# Best Value for k: 17
# F1-Score: 0.7468354430379746

Best K value: 17
Best F1-Score achieved: 0.7468


17

If all went well, you'll notice that model performance has improved by 3 percent by finding an optimal value for k. For further tuning, you can use scikit-learn's built-in `GridSearch()` to perform a similar exhaustive check of hyperparameter combinations and fine tune model performance. For a full list of model parameters, see the [sklearn documentation !](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

## (Optional) Level Up: Iterating on the data

As an optional (but recommended!) exercise, think about the decisions you made during the preprocessing steps that could have affected the overall model performance. For instance, you were asked to replace the missing age values with the column median. Could this have affected the overall performance? How might the model have fared if you had just dropped those rows, instead of using the column median? What if you reduced the data's dimensionality by ignoring some less important columns altogether?

In the cells below, revisit your preprocessing stage and see if you can improve the overall results of the classifier by doing things differently. Consider dropping certain columns, dealing with missing values differently, or using an alternative scaling function. Then see how these different preprocessing techniques affect the performance of the model. Remember that the `find_best_k()` function handles all of the fitting; use this to iterate quickly as you try different strategies for dealing with data preprocessing! 

This is where real data science happens! Hyperparameter tuning (like finding $K=17$) is great, but your model is only ever as good as the data you feed it.Here are three distinct preprocessing experiments you can run right now to see if you can beat that 0.7468 F1-score.Experiment 1: Smart Imputation (Age by Pclass/Sex)Instead of filling missing ages with the global median (~28 years old), we can fill them using the median of a passenger's specific group. For example, a 1st-class passenger was generally older than a 3rd-class passenger.Python# Re-load a fresh copy of your original dataframe first (e.g., df = raw_df.copy())
# Fill missing age based on the median of their Passenger Class and Sex
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
Experiment 2: Dropping Uninformative ColumnsColumns like PassengerId, Ticket, or Name (if left in) just add noise because they are unique to individuals. Even Fare and Pclass sometimes duplicate the same information (wealth). Let's see what happens if we drop unique identifiers before one-hot encoding:Python# Drop columns that don't have general predictive power
df_dropped = df.drop(columns=['PassengerId', 'Name', 'Ticket'])
Experiment 3: Try a Different Scaler (MinMaxScaler)StandardScaler converts data to z-scores, which can technically stretch out infinitely. MinMaxScaler squishes all features strictly between 0 and 1. For KNN, this forces binary one-hot columns (0 or 1) and continuous columns (like age, now 0 to 1) to have the exact same mathematical weight.Pythonfrom sklearn.preprocessing import MinMaxScaler

# Instantiate MinMaxScaler instead
min_max_scaler = MinMaxScaler()

# Scale the data
scaled_data_train_mm = min_max_scaler.fit_transform(X_train)
scaled_data_test_mm = min_max_scaler.transform(X_test)

# Test it immediately with your function!
find_best_k(scaled_data_train_mm, y_train, scaled_data_test_mm, y_test)
How to test your ideas quickly:Pick one (or a combination) of the strategies above, apply it to your dataframe, rerun your train_test_split, scale the data, and pass it directly into your find_best_k() function.See if you can break past 75% F1-Score! Which strategy are you going to try first?

## Summary

Well done! In this lab, you worked with the classic Titanic dataset and practiced fitting and tuning KNN classification models using scikit-learn! As always, this gave you another opportunity to continue practicing your data wrangling skills and model tuning skills using Pandas and scikit-learn!